# OmniVoice Fine-tune 1 Giong Noi (Colab)

Notebook nay giup ban fine-tune 1 giong rieng bang OmniVoice.

## Chuan bi du lieu
- Moi sample gom 1 file wav + 1 file txt cung ten.
- Vi du: `001.wav` va `001.txt`.
- Khuyen dung it nhat 20 sample (cang nhieu cang tot).
- Đat du lieu trong zip theo cau truc:

```
dataset/
  audio/
    001.wav
    002.wav
  text/
    001.txt
    002.txt
```


In [ ]:
#@title 1) Check GPU
!nvidia-smi

In [ ]:
#@title 2) Clone repo + install deps
!git clone https://github.com/k2-fsa/omnivoice.git /content/omnivoice-kit || true
%cd /content/omnivoice-kit
!pip install -r requirements.txt
!pip install accelerate torchvision timm



## 3) Bo buoc upload raw dataset

Flow nay dung `tokens.zip`, khong can upload dataset raw.


## 4) Bo buoc giai nen vao data/raw

Khong su dung `data/raw` khi train tu token da tao san.


## 5) Bo buoc chon raw data

Notebook nay chay theo huong **dung token da tao san** (`tokens.zip`).
Vi vay khong can chon `audio_dir`/`text_dir` tu raw data.



## 6) Bo buoc tao manifest tu raw data

Khi dung `tokens.zip`, manifest/token index da co san nen khong can chay `prepare_manifest.py`.



In [ ]:
#@title 5) Chinh train config cho Colab GPU (fp16, steps)
import json

STEPS = 1200  #@param {type:'integer'}
LEARNING_RATE = 1e-5  #@param {type:'number'}
BATCH_TOKENS = 4096  #@param {type:'integer'}
GRAD_ACC = 2  #@param {type:'integer'}
NUM_WORKERS = 2  #@param {type:'integer'}

cfg_path = '/content/omnivoice-kit/finetune/train_config_finetune_sdpa.json'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

cfg['attn_implementation'] = 'sdpa'
cfg['mixed_precision'] = 'fp16'
cfg['steps'] = int(STEPS)
cfg['learning_rate'] = float(LEARNING_RATE)
cfg['batch_tokens'] = int(BATCH_TOKENS)
cfg['gradient_accumulation_steps'] = int(GRAD_ACC)
cfg['num_workers'] = int(NUM_WORKERS)
cfg['logging_steps'] = 20
cfg['eval_steps'] = 200
cfg['save_steps'] = 200

with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print('Updated config:')
show_keys = [
    'attn_implementation', 'mixed_precision', 'steps', 'learning_rate',
    'batch_tokens', 'gradient_accumulation_steps', 'num_workers',
    'logging_steps', 'eval_steps', 'save_steps'
]
print(json.dumps({k: cfg[k] for k in show_keys}, indent=2))




In [ ]:
#@title 6) RAM-safe env (recommended)
%env TOKENIZERS_PARALLELISM=false
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1

import gc
import os
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Applied RAM-safe env. Ready for Stage 1.")


In [ ]:
#@title 7) Stage 1: Nap tokens.zip va chuan hoa cau truc

import zipfile
import shutil
from pathlib import Path

repo_root = Path('/content/omnivoice-kit')
tokens_root = repo_root / 'data' / 'finetune' / 'tokens'

# Dat duong dan file zip token da upload len Colab
TOKEN_ZIP = '/content/tokens.zip'  #@param {type:'string'}

train_lst = tokens_root / 'train' / 'data.lst'
dev_lst = tokens_root / 'dev' / 'data.lst'

if not TOKEN_ZIP.strip():
    raise ValueError('Hay nhap TOKEN_ZIP, vi flow nay chi dung token da tao san')

zpath = Path(TOKEN_ZIP.strip())
if not zpath.exists():
    raise FileNotFoundError(f'TOKEN_ZIP not found: {zpath}')

if tokens_root.exists():
    shutil.rmtree(tokens_root)
tokens_root.mkdir(parents=True, exist_ok=True)

# Unzip voi normalize slash de tranh loi zip tao tren Windows
with zipfile.ZipFile(zpath, 'r') as zf:
    for info in zf.infolist():
        name = info.filename.replace('\\', '/').lstrip('/')
        if not name:
            continue
        out_path = tokens_root / name
        if name.endswith('/'):
            out_path.mkdir(parents=True, exist_ok=True)
            continue
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info, 'r') as src, open(out_path, 'wb') as dst:
            dst.write(src.read())

print(f'Extracted token zip: {zpath}')

# Tim root hop le co dong thoi train/data.lst va dev/data.lst
if not (train_lst.exists() and dev_lst.exists()):
    detected_root = None
    for cand in tokens_root.rglob('train/data.lst'):
        for anc in [cand.parent.parent] + list(cand.parents):
            if (anc / 'train' / 'data.lst').exists() and (anc / 'dev' / 'data.lst').exists():
                detected_root = anc
                break
        if detected_root is not None:
            break

    if detected_root is not None:
        print(f'Detected token root: {detected_root}')
        if detected_root != tokens_root:
            shutil.copytree(detected_root / 'train', tokens_root / 'train', dirs_exist_ok=True)
            shutil.copytree(detected_root / 'dev', tokens_root / 'dev', dirs_exist_ok=True)
            print('Normalized token structure to /content/omnivoice-kit/data/finetune/tokens/{train,dev}')

if not (train_lst.exists() and dev_lst.exists()):
    raise FileNotFoundError(
        'Khong tim thay train/data.lst va dev/data.lst sau khi unzip. '
        'Hay kiem tra lai cau truc tokens.zip'
    )

# Rewrite data.lst ve path Linux trong Colab

def rewrite_lst(split: str):
    lst = tokens_root / split / 'data.lst'
    lines = [x.strip() for x in lst.read_text(encoding='utf-8').splitlines() if x.strip()]
    new_lines = []
    for ln in lines:
        parts = ln.split()
        if len(parts) < 4:
            continue
        tar_old, jsonl_old, n, sec = parts[0], parts[1], parts[2], parts[3]
        tar_name = Path(tar_old.replace('\\', '/')).name
        jsonl_name = Path(jsonl_old.replace('\\', '/')).name
        tar_new = tokens_root / split / 'audios' / tar_name
        jsonl_new = tokens_root / split / 'txts' / jsonl_name
        new_lines.append(f'{tar_new} {jsonl_new} {n} {sec}')
    lst.write_text('\n'.join(new_lines) + '\n', encoding='utf-8')
    print(f'Rewrote paths: {lst}')

rewrite_lst('train')
rewrite_lst('dev')

print('Token files ready:')
print(' -', train_lst)
print(' -', dev_lst)
print('Stage 1 done. Co the chay Stage 2.')



In [ ]:
#@title 8) Stage 2: Train OmniVoice
import os
from pathlib import Path

repo_root = Path('/content/omnivoice-kit')
tokens_root = repo_root / 'data' / 'finetune' / 'tokens'
train_lst = tokens_root / 'train' / 'data.lst'
dev_lst = tokens_root / 'dev' / 'data.lst'

if not train_lst.exists() or not dev_lst.exists():
    raise FileNotFoundError('Thieu token index. Chay Stage 1 truoc.')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

!accelerate launch --num_processes 1 -m omnivoice.cli.train \
  --train_config /content/omnivoice-kit/finetune/train_config_finetune_sdpa.json \
  --data_config /content/omnivoice-kit/finetune/data_config_finetune.json \
  --output_dir /content/omnivoice-kit/exp/voice_clone_finetune


In [ ]:
#@title 11) Test infer bằng checkpoint mới nhất
import glob
import os

TEST_TEXT = 'Xin chào, đây là giọng nói sau khi fine tune.'  #@param {type:'string'}
REF_AUDIO = ''  #@param {type:'string'}

ckpts = sorted(glob.glob('exp/voice_clone_finetune/checkpoint-*'))
if not ckpts:
    raise RuntimeError('Không tìm thấy checkpoint nào trong exp/voice_clone_finetune')
latest_ckpt = ckpts[-1]
print('Using checkpoint:', latest_ckpt)

if not REF_AUDIO:
    raise RuntimeError('Hãy điền REF_AUDIO (đường dẫn wav tham chiếu) trước khi chạy cell này.')

!python clone_tts.py \
  --model "{latest_ckpt}" \
  --text "{TEST_TEXT}" \
  --ref_audio "{REF_AUDIO}" \
  --output out_finetuned.wav \
  --device cuda

print('Output file: /content/omnivoice-voice-clone-kit/out_finetuned.wav')